![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


In [160]:
# Import necessary packages
import pandas as pd
import torch

import transformers
from transformers import logging
logging.set_verbosity(logging.WARNING)

In [161]:
# Start your code here!

In [162]:
df = pd.read_csv('data/car_reviews.csv', delimiter=";")
df.head()

,Review,Class
0,I am very satisfied with my 2014 Nissan NV SL....,POSITIVE
1,The car is fine. It's a bit loud and not very ...,NEGATIVE
2,"My first foreign car. Love it, I would buy ano...",POSITIVE
3,I've come across numerous reviews praising the...,NEGATIVE
4,I've been dreaming of owning an SUV for quite ...,POSITIVE


In [163]:
from transformers import pipeline

classifier = pipeline(model='distilbert-base-uncased-finetuned-sst-2-english', task='sentiment-analysis')

Device set to use cpu


In [164]:
# Perform inference on the car reviews and display prediction results

# prediction = {'Review':[], 'pred':[]}
# for i,j in zip(df['Review'], df['Class']):
#     prd = classifier(i)
#     prediction['pred'].append(prd[0]['label'])
#     prediction['Review'].append(i)

predicted_labels = classifier(df['Review'].tolist())
pd.DataFrame(predicted_labels).tail()

,label,score
0,POSITIVE,0.929398
1,POSITIVE,0.865427
2,POSITIVE,0.999464
3,NEGATIVE,0.993531
4,POSITIVE,0.998657


In [165]:
# Load accuracy and F1 score metrics

import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [166]:
references = [1 if label == "POSITIVE" else 0 for label in df['Class']]
predictions = [1 if label == "POSITIVE" else 0 for label in predicted_labels]

df['label'] = references
df['pred_label'] = predictions

df.head()

,Review,Class,label,pred_label
0,I am very satisfied with my 2014 Nissan NV SL....,POSITIVE,1,0
1,The car is fine. It's a bit loud and not very ...,NEGATIVE,0,0
2,"My first foreign car. Love it, I would buy ano...",POSITIVE,1,0
3,I've come across numerous reviews praising the...,NEGATIVE,0,0
4,I've been dreaming of owning an SUV for quite ...,POSITIVE,1,0


In [167]:
accuracy_result_dict = accuracy.compute(references=references, predictions=predictions)
accuracy_result = accuracy_result_dict['accuracy']

f1_result_dict = f1.compute(references=references, predictions=predictions)
f1_result = f1_result_dict['f1']

print(accuracy_result)
print(f1_result)

0.4
0.0


## Translation

In [168]:
translator = pipeline(task='translation', model='Helsinki-NLP/opus-mt-en-es')

translated_review = translator(df['Review'][0], max_length=40)[0]['translation_text']
print(f"Model translation:\n{translated_review}")

Device set to use cpu
Your input_length: 365 is bigger than 0.9 * max_length: 40. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Model translation:
Estoy muy satisfecho con mi 2014 Nissan NV SL. Utilizo esta furgoneta para mis entregas de negocios y uso personal. Camping, viajes por carretera, etc. No tenemos niños


In [169]:
# Load reference translations from file
with open("data/reference_translations.txt", 'r') as file:
    lines = file.readlines()
references = [line.strip() for line in lines]
print(f"Spanish translation references:\n{references}")

Spanish translation references:
['Estoy muy satisfecho con mi Nissan NV SL 2014. Utilizo esta camioneta para mis entregas comerciales y uso personal.', 'Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta furgoneta para mis entregas comerciales y uso personal.']


In [170]:
# Load and calculate BLEU score metric
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(predictions=[translated_review], references=[references])

print(bleu_score['bleu'])

0.3515491871310963


## Extractive QA

In [171]:
# Instantiate model and tokenizer
model_ckp = "deepset/minilm-uncased-squad2"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_ckp)
model = transformers.AutoModelForQuestionAnswering.from_pretrained(model_ckp)

# Define context and question, and tokenize them
context = df['Review'][1]
print(f"Context:\n{context}")

question = "What did he like about the brand?"
inputs = tokenizer(question, context, return_tensors="pt")

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Context:
The car is fine. It's a bit loud and not very powerful. On one hand, compared to its peers, the interior is well-built. The transmission failed a few years ago, and the dealer replaced it under warranty with no issues. Now, about 60k miles later, the transmission is failing again. It sounds like a truck, and the issues are well-documented. The dealer tells me it is normal, refusing to do anything to resolve the issue. After owning the car for 4 years, there are many other vehicles I would purchase over this one. Initially, I really liked what the brand is about: ride quality, reliability, etc. But I will not purchase another one. Despite these concerns, I must say, the level of comfort in the car has always been satisfactory, but not worth the rest of issues found.


In [172]:
inputs.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

In [173]:
# Perform inference and extract answer from raw outputs
with torch.no_grad():
  outputs = model(**inputs)

print(outputs)

start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits) + 1
answer_span = inputs["input_ids"][0][start_idx:end_idx]

print(f'start idx : {start_idx}')

QuestionAnsweringModelOutput(loss=None, start_logits=tensor([[ 2.2762, -6.2343, -6.0321, -6.0678, -6.0882, -6.0398, -6.0916, -6.3243,
         -6.6169,  2.2762, -4.1784, -5.3074, -6.0259, -5.0759, -6.3563, -4.7436,
         -6.3526, -6.4298, -5.7884, -5.7856, -5.1914, -6.4162, -4.9813, -5.8880,
         -6.1990, -6.5654, -5.0210, -6.2009, -6.6289, -6.4944, -5.6861, -6.2675,
         -6.1552, -6.4367, -6.4137, -5.1342, -5.3643, -6.2587, -5.5127, -6.2846,
         -6.5086, -6.5867, -5.4555, -5.4227, -6.3117, -6.0545, -6.1566, -6.4791,
         -6.6384, -6.8284, -6.2884, -5.4548, -5.0252, -6.0445, -6.5075, -5.9411,
         -6.0409, -6.8019, -6.3508, -5.8416, -6.7474, -6.7375, -6.0882, -6.3069,
         -5.2681, -5.4771, -6.5445, -6.5645, -6.5778, -6.5040, -5.7578, -5.9364,
         -6.3930, -6.5072, -6.7121, -6.6489, -5.7286, -6.0742, -6.1914, -5.8883,
         -6.2878, -6.8106, -6.4051, -6.0894, -6.1538, -6.4427, -6.1129, -6.5051,
         -6.7101, -6.6582, -3.8438, -4.4204, -5.9236, -5

In [174]:
# Decode and show answer
answer = tokenizer.decode(answer_span)
print("Answer: ", answer)

Answer:  ride quality, reliability


## Get original text to summarize upon car review

In [175]:
text_to_summarize = df['Review'].tolist()[-1]
print(f"Original text:\n{text_to_summarize}")

Original text:
I've been dreaming of owning an SUV for quite a while, but I've been driving cars that were already paid for during an extended period. I ultimately made the decision to transition to a brand-new car, which, of course, involved taking on new payments. However, given that I don't drive extensively, I was inclined to avoid a substantial financial commitment. The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment; the financial arrangement is quite reasonable. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. I am VERY satisfied overall. I find myself needing to exercise extra caution when making lane changes, particularly owing to the blind spots resulting from the small side windows situated towards the rear of the vehicle. To address this concern, I am actively engaged in making adjustments to my mirrors and consciously reducing the frequency of la

In [176]:
# Load summarization pipeline and perform inference
model_name = "cnicu/t5-small-booksum"
summarizer = pipeline("summarization", model=model_name)

outputs = summarizer(text_to_summarize, max_length=53)
summarized_text = outputs[0]['summary_text']

print(f"Summarized text:\n{summarized_text}")

Device set to use cpu


Summarized text:
the Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment; the financial arrangement is quite reasonable. I have hauled 12 bags of mulch in the back with the seats down and could have held more.
